# 01 — Environment setup (Apple Silicon M4)

**Goal.** Bootstrap a reproducible Python 3.11 environment that runs Gymnasium 1.0 (Box2D), Stable-Baselines3 2.4, and PyTorch with the **Metal Performance Shaders (MPS)** backend enabled.

**Why this matters for the thesis.** Project 50.1 will run *many* multi-seed training jobs on a fanless M4. Two constraints drive every choice in this notebook:

1. **Env stepping stays on CPU.** Box2D physics is single-threaded; the M4 P-cores beat any GPU/MPS hop for small obs/action vectors.
2. **Neural-net training goes to MPS** when the network is large enough to amortise per-call dispatch latency (SAC critics, future LSTM forward-models).

We codify this split in `src/utils/device.py`.

---

## Bugs / pitfalls encountered while setting this up

| # | Issue | Cause | Fix |
|---|---|---|---|
| 1 | `pip install gymnasium[box2d]` fails on a fresh macOS | `swig` (Box2D wheel build dep) not on PATH | `brew install swig` *before* installing requirements |
| 2 | `torch.backends.mps.is_available() == False` on macOS | Installed wrong wheel via Anaconda x86 | Use python.org / Homebrew arm64 Python 3.11, then `pip install torch==2.4.1` |
| 3 | Slow PPO when forced onto MPS | MPS dispatch latency dominates the small MLP forward pass | Pin PPO to CPU; reserve MPS for SAC / forward-models |

## 1.1 Install dependencies

Run the cell below **once** at the start of a session if you have not already executed `setup.sh`. Inside a fresh shell:

```bash
cd code/py
./setup.sh                 # creates .venv, installs everything
source .venv/bin/activate
jupyter lab                # then re-open this notebook in the .venv kernel
```

In [1]:
# Optional: install in-place if you skipped setup.sh.
# Comment out once you have a working .venv to avoid re-installing each kernel restart.
#!pip install -q -r ../py/requirements.txt

## 1.2 Verify the toolchain

We want to see:

- `torch.__version__ == '2.4.1'`
- `torch.backends.mps.is_available() == True`
- `gymnasium.__version__ == '1.0.0'`
- `stable_baselines3.__version__ == '2.4.0'`

If MPS reports `False`, the rest of the project will silently fall back to CPU. That's *correct* behaviour for PPO but a regression for SAC.

In [2]:
import platform, sys
import torch
import gymnasium as gym
import stable_baselines3 as sb3

print(f'python       : {sys.version.split()[0]}')
print(f'platform     : {platform.platform()}  ({platform.machine()})')
print(f'torch        : {torch.__version__}')
print(f'  mps_built  : {torch.backends.mps.is_built()}')
print(f'  mps_avail  : {torch.backends.mps.is_available()}')
print(f'  cuda_avail : {torch.cuda.is_available()}')
print(f'gymnasium    : {gym.__version__}')
print(f'sb3          : {sb3.__version__}')

python       : 3.11.14
platform     : macOS-26.5.1-arm64-arm-64bit  (arm64)
torch        : 2.4.1
  mps_built  : True
  mps_avail  : True
  cuda_avail : False
gymnasium    : 1.0.0
sb3          : 2.4.0


## 1.3 Wire the project source tree onto `sys.path`

Notebooks live in `code/ipynb/`; the library code lives in `code/py/src/`. Prepend the sibling `py/` directory so `from src.envs...` and `from src.utils...` work.

In [3]:
import sys, pathlib
PY_ROOT = pathlib.Path('..').resolve() / 'py'
if str(PY_ROOT) not in sys.path:
    sys.path.insert(0, str(PY_ROOT))
print(f'src root added to sys.path: {PY_ROOT}')
assert (PY_ROOT / 'src' / 'envs' / 'wrappers.py').exists(), 'cannot find py/src — check working directory'

src root added to sys.path: /Users/raghuramantm/Desktop/Thesis Proposal/code/py


## 1.4 Device-selection sanity check

`get_torch_device('auto')` should return `mps` on the M4. We also cap BLAS threads — important because `SubprocVecEnv` will spawn multiple env workers and we do **not** want them fighting over the same P-cores.

In [4]:
from src.utils.device import get_torch_device, configure_threading, describe_device

configure_threading(num_threads=1)
device = get_torch_device('auto')
print(describe_device(device))

device=mps | torch=2.4.1 | mps_built=True | mps_avail=True | cuda_avail=False


## 1.5 Seed the world

Multi-seed evaluation is non-optional for publication-grade RL. We use the convention seeds = `[0,1,2,3,4]` (Henderson et al., 2018 — *Deep Reinforcement Learning that Matters*).

In [5]:
from src.utils.seeding import set_global_seed, seed_list

set_global_seed(0)
print('multi-seed schedule:', seed_list(0, 5))

multi-seed schedule: [0, 1, 2, 3, 4]


**Checkpoint.** If all cells executed with no exceptions and MPS shows as available, proceed to notebook **02 — envs and wrappers**.